In [1]:
import torch
import sys
from models.forecasting.diffusion import DDPM
from models.unet.unet import UNet
from models.model_utils.noise_scheduler import CosineScheduler

In [2]:
num_pde_coeff = 2
unet_base_out_channels = 4
unet_in_channels = num_pde_coeff + 1
kernel_size = 3
final_filters = 1
encoder_dropout = 0
input_size = 256
num_diffusion_steps = 10
cosine_shift = 0.01
device = torch.device('mps')

unet = UNet(
        in_channels = unet_in_channels,
        out_channels = unet_base_out_channels,
        kernel_size = kernel_size,
        final_filters = final_filters,
        encoder_dropout = encoder_dropout,
        input_size = input_size,
    ).to(device)
scheduler = CosineScheduler(T = num_diffusion_steps, s = cosine_shift)
noise_schedule = scheduler.schedule().to(device)

ddpm = DDPM(denoising_model = unet, noise_schedule = noise_schedule).to(device)

In [3]:
noise_schedule

tensor([1.0000e+00, 9.7125e-01, 8.9729e-01, 7.8521e-01, 6.4576e-01, 4.9234e-01,
        3.3967e-01, 2.0239e-01, 9.3694e-02, 2.3999e-02, 1.9111e-15],
       device='mps:0')

In [4]:
batch_size = 64
t = torch.randn((batch_size,)).to(device)
mock_backbone_input = torch.randn((batch_size, unet_in_channels, input_size, input_size)).to(device)
mock_diffusion_input = torch.randn((batch_size, num_pde_coeff, input_size // 2, input_size // 2)).to(device)

In [5]:
unet(mock_backbone_input, t).shape

torch.Size([64, 1, 256, 256])

In [6]:
mock_diffusion_input.shape

torch.Size([64, 2, 128, 128])